# Prompt Guard — Regex Baseline

This baseline detects structured sensitive information in a prompt and returns a policy decision
(**allow / warn / block**) and the location of each match. The detector uses hand-written regular
expressions and format checks. There is no model training.

1. **Find candidates** in the raw prompt using regular expressions.
2. **Check the format** where it helps distinguish sensitive data from ordinary numbers.
3. **Apply the shared policy** from config/policy.yaml. The most severe match determines the decision.

| Rule | Examples of supported formats | Dataset label |
|---|---|---|
| E-mail | anna.schmidt@example.com, alex+work@example.org | EMAIL |
| Phone | +49 7131 123456, 07131/123456, (202) 555-0148 | PHONENUMBER |
| IBAN | Compact or groups of four, with a valid checksum | IBAN |
| Credit card | 13–19 digits, compact or grouped, with a valid checksum | CREDITCARDNUMBER |
| Domestic account number | 5–17 consecutive digits following an account-number keyword | ACCOUNTNUMBER |

Open this notebook from the project's notebooks/ folder and select the project Python environment.
The synthetic examples run with the policy alone. The validation and MLflow sections also need the
local dataset files and the committed data/splits.csv. The validation split and metrics match the
TF-IDF + Logistic Regression notebook. The test split is reserved for the team's final comparison.

## 1. Setup and shared policy

The project helper supplies the label mapping and the decision function. The notebook can be
started with the project root or the notebooks directory as the working directory.

In [ ]:
import re
import sys
from hashlib import sha256
from pathlib import Path

from IPython.display import display
import pandas as pd
import yaml

working_dir = Path.cwd().resolve()
PROJECT_DIR = next(
    (folder for folder in (working_dir, *working_dir.parents)
     if (folder / "src" / "data_utils.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Open AIOps_PromptGuard in VS Code and run this notebook from its notebooks/ folder."
    )

sys.path.insert(0, str(PROJECT_DIR / "src"))
from data_utils import load_policy, load_split, prompt_decision, split_fingerprint

CLASSES = ["allow", "warn", "block"]
SEED = 42
RULESET_VERSION = "1.0"

policy = load_policy()
policy_bytes = (PROJECT_DIR / "config" / "policy.yaml").read_bytes()
policy_file = yaml.safe_load(policy_bytes)
POLICY_SHA256 = sha256(policy_bytes).hexdigest()
POLICY_VERSION = (
    policy_file.get("version") or policy_file.get("policy_version")
    or f"sha256:{POLICY_SHA256[:12]}"
)
print(f"Policy version: {POLICY_VERSION}")

## 2. Define the detection rules

Phone candidates need a recognisable phone format or a nearby phone keyword. An account
number needs an explicit account keyword because its digits can also look like an order ID.
The IBAN rule covers DE, FR, IT and six other common European formats. IBAN and card
checksums help reject ordinary strings with a similar shape.

The policy gets the exact detected label and the original character offsets. The end offset
is exclusive, so prompt[start:end] gives the matched value.

In [ ]:
HSPACE = r"[ \u00a0\u202f]"
PHONE_SEP = r"[ .\u00a0\u202f/-]"
NUMBER_START = r"(?<![\w+])(?<![0-9][./-])"
NUMBER_END = rf"(?![\w]|[./-][0-9]|{HSPACE}[0-9])"
VALUE_PREFIX = r"[^\S\r\n]*(?:(?:lautet|ist|is|est|è)\b[^\S\r\n]*)?[:=#]?[^\S\r\n]*"

EMAIL_PATTERN = re.compile(
    r"(?<![\w.%+@-])"
    r"(?P<value>[A-Za-z0-9_%+'-]+(?:\.[A-Za-z0-9_%+'-]+)*@"
    r"(?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+[A-Za-z]{2,63})"
    r"(?![\w@-]|\.[\w-])"
)

IBAN_LENGTHS = {
    "AT": 20, "BE": 16, "CH": 21, "DE": 22, "ES": 24,
    "FR": 27, "GB": 22, "IT": 27, "NL": 18,
}
iban_formats = []
for country, length in IBAN_LENGTHS.items():
    full_groups, remainder = divmod(length - 4, 4)
    tail = rf"(?:{HSPACE}?[A-Z0-9]{{4}}){{{full_groups}}}"
    if remainder:
        tail += rf"{HSPACE}?[A-Z0-9]{{{remainder}}}"
    iban_formats.append(rf"{country}[0-9]{{2}}{tail}")

iban_body = "|".join(iban_formats)
IBAN_PATTERN = re.compile(rf"(?<!\w)(?P<value>(?ai:{iban_body}))(?!\w)")

ACCOUNT_PATTERN = re.compile(
    r"\b(?:kontonummer|konto-?nr\.?|account\s+(?:number|no\.?)|"
    r"num[ée]ro\s+de\s+compte|numero\s+(?:di\s+)?conto)"
    rf"{VALUE_PREFIX}(?P<value>[0-9]{{5,17}})(?![\w]|[./-][0-9])",
    re.IGNORECASE,
)

CARD_PATTERN = re.compile(
    rf"{NUMBER_START}(?P<value>[0-9](?:[ \u00a0\u202f-]?[0-9]){{12,18}}){NUMBER_END}"
)

PHONE_CONTEXT = re.compile(
    r"\b(?:tel\.?|telefon(?:nummer)?|rufnummer|handy(?:nummer)?|"
    r"phone(?:\s+number)?|mobile(?:\s+number)?|t[ée]l[ée]phone|"
    r"num[ée]ro\s+de\s+t[ée]l[ée]phone|cellulare|telefono|numero\s+di\s+telefono)"
    rf"{VALUE_PREFIX}$",
    re.IGNORECASE,
)
PHONE_PATTERN = re.compile(
    rf"{NUMBER_START}(?P<value>(?P<international>\+|00(?=[1-9]))?"
    rf"(?P<main>(?:(?:[0-9]{{1,3}}{PHONE_SEP}?)?\([0-9]{{1,5}}\){PHONE_SEP}?)?"
    rf"[0-9](?:{PHONE_SEP}?[0-9]){{5,16}})"
    rf"(?:{HSPACE}*(?:x|ext\.?){HSPACE}*[0-9]{{1,5}})?){NUMBER_END}",
    re.IGNORECASE,
)
DATE_SHAPE = re.compile(
    r"(?:[0-9]{4}[./-][0-9]{1,2}[./-][0-9]{1,2}|"
    r"[0-9]{1,2}[./-][0-9]{1,2}[./-][0-9]{4})"
)
NORTH_AMERICAN_PHONE = re.compile(
    rf"(?:1{PHONE_SEP})?(?:\([2-9][0-9]{{2}}\){PHONE_SEP}?|[2-9][0-9]{{2}}{PHONE_SEP})"
    rf"[2-9][0-9]{{2}}{PHONE_SEP}[0-9]{{4}}"
)
NON_DIGIT = re.compile(r"[^0-9]")

In [ ]:
def valid_iban(match):
    iban = re.sub(HSPACE, "", match["value"]).upper()
    rotated = iban[4:] + iban[:4]
    numeric = "".join(char if char.isdigit() else str(ord(char) - 55) for char in rotated)
    return 2 <= int(iban[2:4]) <= 98 and int(numeric) % 97 == 1


def valid_card(match):
    digits = NON_DIGIT.sub("", match["value"])
    if digits[0] == "0" or len(set(digits)) == 1:
        return False
    total = 0
    for index, char in enumerate(reversed(digits)):
        digit = int(char)
        if index % 2:
            digit *= 2
            if digit > 9:
                digit -= 9
        total += digit
    return total % 10 == 0


def plausible_phone(match):
    main = match["main"]
    international = match["international"]
    digits = NON_DIGIT.sub("", main.replace("(0)", "", 1) if international else main)
    if not 7 <= len(digits) <= 15 or DATE_SHAPE.fullmatch(main):
        return False

    left_context = match.string[max(0, match.start() - 80):match.start()]
    return (
        bool(international and digits[0] != "0")
        or PHONE_CONTEXT.search(left_context) is not None
        or NORTH_AMERICAN_PHONE.fullmatch(main) is not None
        or (digits.startswith("0") and 9 <= len(digits) <= 12)
    )


# Prefer specific entities when rules overlap. A phone-like substring of an IBAN
# or a clearly labelled account number belongs to that bank detail.
RULES = [
    ("EMAIL", EMAIL_PATTERN, None),
    ("IBAN", IBAN_PATTERN, valid_iban),
    ("ACCOUNTNUMBER", ACCOUNT_PATTERN, None),
    ("CREDITCARDNUMBER", CARD_PATTERN, valid_card),
    ("PHONENUMBER", PHONE_PATTERN, plausible_phone),
]
RULE_LABELS = [label for label, _, _ in RULES]
missing_labels = set(RULE_LABELS) - set(policy["label_to_decision"])
if missing_labels:
    raise ValueError(f"These regex labels are missing from the project policy: {sorted(missing_labels)}")


def detect_pii(prompt):
    spans = []
    for label, pattern, validator in RULES:
        for match in pattern.finditer(prompt):
            if validator is not None and not validator(match):
                continue
            start, end = match.span("value")
            if any(start < span["end"] and span["start"] < end for span in spans):
                continue
            spans.append({"start": start, "end": end, "label": label, "value": match["value"]})
    return sorted(spans, key=lambda span: span["start"])


def predict_prompt(prompt):
    spans = detect_pii(prompt)
    return prompt_decision(spans, policy), spans


display(pd.Series({label: policy["label_to_decision"][label] for label in RULE_LABELS},
                  name="Project policy decision").to_frame())

## 3. Try the baseline on synthetic prompts

These examples check the rules, including one prompt with both an e-mail and an IBAN.
You can call predict_prompt with a new prompt to see its decision and spans.

In [ ]:
examples = [
    ("Fasse bitte unsere neue Urlaubsregelung zusammen.", []),
    ("Schreibe an anna.schmidt@example.com.", ["EMAIL"]),
    ("Schreib an alex+arbeit@example.org!", ["EMAIL"]),
    ("Ruf bitte unter +49 (0)7131 123456 an.", ["PHONENUMBER"]),
    ("Telefon: 07131/123456.", ["PHONENUMBER"]),
    ("Please call (202) 555-0148.", ["PHONENUMBER"]),
    ("Bitte überweise den Betrag an DE89370400440532013000.", ["IBAN"]),
    ("Sende anna.schmidt@example.com die IBAN DE89 3704 0044 0532 0130 00.", ["EMAIL", "IBAN"]),
    ("Kontonummer: 1234567890.", ["ACCOUNTNUMBER"]),
    ("Numéro de compte : 1234567890.", ["ACCOUNTNUMBER"]),
    ("Carta: 4111 1111 1111 1111.", ["CREDITCARDNUMBER"]),
    ("Umsatz 2023, Ticket 12345678, Termin 01.12.2026.", []),
    ("Beispielwerte mit Tippfehlern: DE89370400440532013001 und 4111-1111-1111-1112.", []),
]

example_rows = []
for prompt, expected_labels in examples:
    decision, spans = predict_prompt(prompt)
    assert [span["label"] for span in spans] == expected_labels, (prompt, spans)
    assert all(prompt[span["start"]:span["end"]] == span["value"] for span in spans)
    expected_decision = max(
        (policy["label_to_decision"][label] for label in expected_labels),
        key=CLASSES.index,
        default="allow",
    )
    assert decision == expected_decision, (prompt, decision, expected_decision)
    example_rows.append({
        "prompt": prompt,
        "decision": decision,
        "labels": ", ".join(span["label"] for span in spans) or "—",
        "matches": ", ".join(span["value"] for span in spans) or "—",
    })

assert predict_prompt("")[0] == "allow"
with pd.option_context("display.max_colwidth", 90):
    display(pd.DataFrame(example_rows))

## 4. Run on the shared validation split

Prediction reads only source_text. The decision and privacy_mask columns are used after
prediction to calculate metrics. The split fingerprint identifies the exact split file.
For comparison with the saved TF-IDF results, check for fingerprint 6afd6a257de1 and policy 0.1.

In [ ]:
import json
import time
from collections import Counter

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score

SPLIT_FINGERPRINT = split_fingerprint()
val = load_split("validation")
print(f"Split fingerprint: {SPLIT_FINGERPRINT} | Policy: {POLICY_VERSION}")
print(f"Validation prompts: {len(val):,}")

predictions = [predict_prompt(prompt) for prompt in val["source_text"]]
validation = val.assign(
    predicted_decision=[decision for decision, _ in predictions],
    predicted_spans=[spans for _, spans in predictions],
)

## 5. Evaluate prompt decisions

**Macro F1** gives allow, warn and block the same weight. **Recall block** measures the
share of true block prompts that were blocked. The report uses the full project policy,
including cases whose PII types are outside the regex rules. The confusion matrix shows
counts with true classes in the rows and predictions in the columns.

In [ ]:
def decision_metrics(frame):
    actual = frame["decision"]
    predicted = frame["predicted_decision"]
    return {
        "macro_f1": f1_score(actual, predicted, labels=CLASSES, average="macro", zero_division=0),
        **{
            f"recall_{label}": recall_score(
                actual, predicted, labels=[label], average="macro", zero_division=0
            )
            for label in CLASSES
        },
    }


overall = decision_metrics(validation)
by_language = {
    language: {"prompts": len(part), **decision_metrics(part)}
    for language, part in validation.groupby("language")
}
decisions = pd.DataFrame.from_dict(
    {"overall": {"prompts": len(validation), **overall}, **by_language},
    orient="index",
)
display(decisions.round(4))

actual = validation["decision"]
predicted = validation["predicted_decision"]
print(classification_report(actual, predicted, labels=CLASSES, digits=3, zero_division=0))

confusion = pd.DataFrame(
    confusion_matrix(actual, predicted, labels=CLASSES),
    index=pd.Index(CLASSES, name="True decision"),
    columns=pd.Index(CLASSES, name="Predicted decision"),
)
display(confusion)

## 6. Evaluate the detected text spans

A true positive needs the same start offset, end offset and label as the annotation.
A partial match counts as one false positive and one false negative. This table covers the
five types targeted by the regex rules. The prompt metrics above cover all policy types.

In [ ]:
def span_keys(mask):
    if isinstance(mask, str):
        mask = json.loads(mask)
    return {
        (int(span["start"]), int(span["end"]), span["label"])
        for span in mask
        if span["label"] in RULE_LABELS
    }


def span_metrics(frame):
    tp, fp, fn = Counter(), Counter(), Counter()
    for gold_mask, predicted_mask in zip(frame["privacy_mask"], frame["predicted_spans"]):
        gold = span_keys(gold_mask)
        predicted = span_keys(predicted_mask)
        tp.update(label for _, _, label in gold & predicted)
        fp.update(label for _, _, label in predicted - gold)
        fn.update(label for _, _, label in gold - predicted)

    rows = []
    for label in RULE_LABELS:
        correct, extra, missed = tp[label], fp[label], fn[label]
        support = correct + missed
        n_predicted = correct + extra
        rows.append({
            "label": label,
            "support": support,
            "TP": correct,
            "FP": extra,
            "FN": missed,
            "precision": correct / n_predicted if n_predicted else 0.0,
            "recall": correct / support if support else 0.0,
            "F1": 2 * correct / (n_predicted + support) if n_predicted + support else 0.0,
        })
    return pd.DataFrame(rows).set_index("label")


span_overall = span_metrics(validation)
span_de = span_metrics(validation.loc[validation["language"] == "de"])
display(pd.concat({"overall": span_overall, "de": span_de}, names=["scope"]).round(4))

## 7. Inspect missed blocks and false alarms

The first table counts block-labelled annotations in prompts the baseline missed. A prompt
can contain several labels. The second table counts the labels that triggered a block in
prompts whose reference decision was allow or warn.

In [ ]:
missed_blocks = validation.loc[(actual == "block") & (predicted != "block")]
false_blocks = validation.loc[(actual != "block") & (predicted == "block")]
print(f"Missed blocks: {len(missed_blocks):,} of {(actual == 'block').sum():,} true block prompts")
print(f"False blocks: {len(false_blocks):,} of {(actual != 'block').sum():,} true allow/warn prompts")


def count_block_labels(masks):
    counts = Counter()
    for mask in masks:
        if isinstance(mask, str):
            mask = json.loads(mask)
        counts.update(
            span["label"] for span in mask
            if policy["label_to_decision"].get(span["label"]) == "block"
        )
    return pd.Series(dict(counts), dtype="int64").sort_values(ascending=False)


display(count_block_labels(missed_blocks["privacy_mask"]).to_frame("annotations in missed blocks"))
display(count_block_labels(false_blocks["predicted_spans"]).to_frame("matches in false blocks"))

## 8. Measure response time

Time one prompt at a time, including rule matching and the policy decision. Use the same
sample size and seed as the TF-IDF notebook. Compare latency on the same machine.

In [ ]:
sample = validation["source_text"].sample(min(1000, len(validation)), random_state=SEED)
times_ms = []
for prompt in sample:
    start = time.perf_counter()
    predict_prompt(prompt)
    times_ms.append((time.perf_counter() - start) * 1000)

LATENCY_MEDIAN_MS = float(np.median(times_ms))
LATENCY_P95_MS = float(np.percentile(times_ms, 95))
print(f"Median: {LATENCY_MEDIAN_MS:.3f} ms per prompt")
print(f"95th percentile: {LATENCY_P95_MS:.3f} ms per prompt")

## 9. Log the validation run to MLflow

The local promptguard experiment uses the same metric names and tracking database as
the TF-IDF baseline. The notebook contains the detector. MLflow records the rule version,
policy, split, metrics and the validation tables.

In [ ]:
import mlflow

mlflow.set_tracking_uri(f"sqlite:///{(PROJECT_DIR / 'mlflow.db').as_posix()}")
if mlflow.get_experiment_by_name("promptguard") is None:
    mlflow.create_experiment("promptguard", artifact_location=(PROJECT_DIR / "mlruns").as_uri())
mlflow.set_experiment("promptguard")

with mlflow.start_run(run_name="regex_baseline"):
    mlflow.log_params({
        "model": "regex",
        "ruleset_version": RULESET_VERSION,
        "labels": ", ".join(RULE_LABELS),
        "split_fingerprint": SPLIT_FINGERPRINT,
        "policy_version": str(POLICY_VERSION),
        "policy_sha256": POLICY_SHA256,
    })
    mlflow.log_metrics({
        **{f"val_{metric}": value for metric, value in overall.items()},
        "val_de_macro_f1": by_language["de"]["macro_f1"],
        "val_de_recall_block": by_language["de"]["recall_block"],
        "latency_median_ms": LATENCY_MEDIAN_MS,
        "latency_p95_ms": LATENCY_P95_MS,
    })
    mlflow.log_dict(
        {
            "decision_metrics": json.loads(decisions.to_json(orient="index")),
            "confusion_matrix": json.loads(confusion.to_json(orient="index")),
            "exact_spans_overall": json.loads(span_overall.to_json(orient="index")),
            "exact_spans_de": json.loads(span_de.to_json(orient="index")),
        },
        "validation_metrics.json",
    )
print("Regex baseline logged to MLflow.")

## 10. Interpret the results

- Compare validation **Macro F1** and **Recall block** with the TF-IDF baseline for all languages and for German.
- Use the span table to see which structured PII types the rules can locate. A correctly located e-mail can still yield a missed block if the same prompt contains a password outside the rules.
- Names, addresses, passwords, PINs and obfuscated values are outside the baseline's coverage. Unlabelled account numbers and some phone formats can also be missed. Checksum rules can miss annotated values that fail the format check.
- Keep the team's final test evaluation and separate realistic German test set for the final model comparison.

**References**

- [Dataset labels](https://huggingface.co/datasets/ai4privacy/pii-masking-200k/blob/main/pii_class_counts.json) and [character-span schema](https://huggingface.co/datasets/ai4privacy/pii-masking-200k/blob/main/README.md)
- [SWIFT IBAN Registry](https://www.swift.com/swift-resource/9606/download)
- [IBM documentation on the card check digit](https://www.ibm.com/docs/en/iotdm/11.7.0?topic=providers-credit-card-privacy-provider)